# Calculating the Equilibrium Poplulations

In [ ]:
import itertools
import math
import random
from collections import Counter, defaultdict, OrderedDict
from itertools import combinations, islice
from math import nan, isnan
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy.linalg import expm, eig, norm
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.patches as mpatches
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler, normalize
import networkx as nx
from community import community_louvain
import community as c  # You may only need one of these
import MDAnalysis as mda
from MDAnalysis.coordinates.XTC import XTCWriter
import mdtraj as md
import pickle
np.set_printoptions(legacy='1.25')


In [88]:
alpha_full_length = "alpha_syn_full_peptide"
medin_cm14 = 'medin_cm14'
medin_cm8 = 'medin_cm8'
abeta = "abeta"
medin_urea = 'medin_urea'

protein_name = medin_cm8


In [89]:
def process_weights(w_file):
    KBT = 2.49  # kJ/mol
    colvar_file = w_file
    data = np.loadtxt(colvar_file, comments="#")
    bias = data[:, -1]
    weights = np.exp(bias / KBT)
    weights /= weights.sum() 

def trim(file,n_reps,trim_fraction):
    w_split = np.array_split(file, n_reps)
    trimmed_chunks = [chunk[int(trim_fraction * len(chunk)):] for chunk in w_split]
    w_trimmed = np.concatenate(trimmed_chunks)
    return(w_trimmed)

In [90]:
# com_path = f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/{protein_name}/d_24_t_com_avg.pkl'
# with open(com_path, 'rb') as file_com:
#     distances_com = pickle.load(file_com)

# closest_path = f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/{protein_name}/d_24_t_closest.pkl'
# with open(closest_path, 'rb') as file_closest:
#     distances_closest = pickle.load(file_closest)

In [91]:
if protein_name == 'abeta':
    pdb = '/Users/adelielouet/Documents/science/AB_G5_original_simu_analysis/trajectories/Gabis_paper/template_G5.pdb'
    xtc = '/Users/adelielouet/Documents/science/AB_G5_original_simu_analysis/trajectories/Gabis_paper/traj_all-skip-0-noW_G5.xtc'
    w_file = "/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/abeta/weights/weights_corr_G5"
    distances_com = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/abeta/distances/d_24_t_com_avg.pkl', 'rb'))
    distances_closest = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/abeta/distances/d_24_t_com_avg.pkl', 'rb'))
    w_eq = np.loadtxt(w_file)
    resname_ligand = "liga"
    special_char = 'A\u03B2'
    w_com = 0.8
    w_closest = 0.2

elif protein_name == 'medin_cm14':
    pdb = '/Users/adelielouet/Documents/science/medin/cm14/simu/prepared_system/CM14/prepared_tpr_files_v1/template_complex.gro'
    xtc = '/Users/adelielouet/Documents/science/medin/cm14/simu/prepared_system/CM14/prepared_tpr_files_v1/cat_trjcat.xtc'
    resname_ligand = "1UNL"
    special_char = 'Medin_cm14'
    w_com = 0.45
    w_closest = 0.55
    w_eq=np.array([1.0] *len(distances_com))   # this is for systems that don't need to be reweighted, assign each frame a weight of 1.0

elif protein_name == 'medin_cm8':
    pdb = '/Users/adelielouet/Documents/science/medin/cm8/cm8_post_run/protein_ligand.gro'
    xtc = '/Users/adelielouet/Documents/science/medin/cm8/cm8_post_run/concatenate_cm8_skip_10.xtc'
    w_file='/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_cm8/weights/COLVAR_REWEIGHT'
    distances_com = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_cm8/distances/d_24_t_com_avg.pkl', 'rb'))
    distances_closest = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_cm8/distances/d_24_t_closest.pkl', 'rb'))
    resname_ligand = "1UNL"
    special_char = 'Medin_cm8'
    w_eq = process_weights(w_file)
    w_com = 0.45
    w_closest = 0.55
    w_eq=np.array([1.0] *len(distances_com))

elif protein_name == 'alpha_syn_full_peptide':
    pdb = '/Users/adelielouet/Documents/science/alpha_syn/DE_SHAW/system.pdb'
    # xtc = ???  # xtc path appears to be missing for alpha
    resname_ligand = "*"
    special_char = "\u03B1-syn-full"
    w_com = 0.1
    w_closest = 0.9
    w_eq=[1.0] *len(distances_com)   # this is for systems that don't need to be reweighted, assign each frame a weight of 1.0

elif protein_name == 'medin_urea':
    pdb = '/Users/adelielouet/Documents/science/medin/urea/urea_post_run/protein_urea.gro'
    xtc = '/Users/adelielouet/Documents/science/medin/urea/urea_post_run/concatenate_urea.xtc'
    resname_ligand = "1UNL"
    distances_com = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_urea/distances/d_24_t_com_avg.pkl', 'rb'))
    distances_closest = pickle.load(open(f'/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_urea/distances/d_24_t_closest.pkl', 'rb'))
    w_file='/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/medin_cm8/weights/COLVAR_REWEIGHT'
    w_eq = process_weights(w_file)
    special_char = 'Medin_urea'
    w_com = 0.45
    w_closest = 0.55
    w_eq=np.array([1.0] *len(distances_com))

else:
    pass


## Post-processing Trajectories

This includes treatign the weights "pbmetad.bias" as well as removing first 10% of each trajecotry (whcih for 16 concatenated replicas means divinding files by 16, removing first 10% and reconcatentating)

In [92]:
w_eq /= w_eq.sum()  # Normalize so weights sum to 1
distances_combined = (w_com * np.array(distances_com)) + (w_closest * np.array(distances_closest))
distance_threshold_combined = (0.75 * w_com) + (0.45 * w_closest)
number_contact = np.array([np.sum(values <= distance_threshold_combined) for values in distances_combined])

##########

bound_mask = number_contact != 0
unbound_mask = ~bound_mask

avg_contacts_when_bound = np.sum(w_eq[bound_mask] * number_contact[bound_mask]) / np.sum(w_eq[bound_mask])

fraction_bound_weighted = np.sum(w_eq[bound_mask])

total_bound_weighted = np.sum(w_eq[bound_mask]) * len(number_contact)


In [93]:
fraction_bound_weighted

0.9278492294772284

## Generating Transition Matrix

In [ ]:
### This is where you decide the -uplet type
uplet_type=5

u = mda.Universe(pdb)#, xtc)

protein_residues = u.select_atoms("protein").residues
num_residues = len(protein_residues)

residue_pairs = list(combinations(range(num_residues), uplet_type))  # all unique pairs of residues
contact_counts_top_uplet_type_indices = {pair: 0 for pair in residue_pairs}


distances=distances_combined
num_timesteps = distances.shape[0]

contact_counts_uplet_type_timesteps=[]
contact_counts_uplet_type_timesteps_inlcuding_0=[]

w_eq_filtered_zeros=[]
for t,w in zip(range(num_timesteps),w_eq):
    if w!=0.0:
        close_residues = np.where(distances[t, :] < distance_threshold_combined)[0]
        close_residues_values = [x for x in distances[t] if x < distance_threshold_combined]

        paired = list(zip(close_residues_values, close_residues))
        sorted_pairs = sorted(paired, key=lambda x: x[0])
        top_uplet_type_indices = [pair[1] for pair in sorted_pairs[:uplet_type]]
    # print(close_residues)
        if len(top_uplet_type_indices) == uplet_type:
            contact_counts_uplet_type_timesteps.append(top_uplet_type_indices)
            contact_counts_uplet_type_timesteps_inlcuding_0.append(top_uplet_type_indices)
            w_eq_filtered_zeros.append(w)
        else:
            contact_counts_uplet_type_timesteps_inlcuding_0.append(([-1]*uplet_type))
    else:
        print('no state exists when weighted')
        contact_counts_uplet_type_timesteps_inlcuding_0.append(([-1]*uplet_type))

#this sorts the values of sublists in order so that we don't have any repeats ([6,3,9) =[3,6,9]]) 
unique_uplets_pre_process=[sorted(sublist) for sublist in contact_counts_uplet_type_timesteps]
unique_uplets_pre_process_0=[sorted(sublist) for sublist in contact_counts_uplet_type_timesteps_inlcuding_0]

frequency = Counter(tuple(sorted(sublist)) for sublist in unique_uplets_pre_process)
print(f'For {uplet_type}, there are {len(frequency)} unique pairs')


filtered_keys=[item for item, count in frequency.items()]

data_preprocessed = [tuple(x) for x in unique_uplets_pre_process]

data = [sublist for sublist in data_preprocessed if sublist in filtered_keys]#[filtered_keys]
value_to_index =  {tuple(row): index for index, row in enumerate(filtered_keys)}#(filtered_keys)

print(len(filtered_keys),len(value_to_index))

### Need for an unbound state to avoid absorbing states- start with unbound and edn with undbound, or else risk having the leaving state absorbed
unbound_state = tuple([0] * uplet_type)
data.insert(0, unbound_state)
data.append(unbound_state)
frequency[unbound_state]=2
filtered_keys.append(unbound_state)
value_to_index[unbound_state]=len(value_to_index)

print(len(filtered_keys),len(value_to_index))


transition_matrix = np.zeros((len(filtered_keys), len(filtered_keys)), dtype=np.float64)

print(len(transition_matrix))

w_eq_filtered_zeros.insert(0, min(w_eq_filtered_zeros))  # serves to add the lowest weight possible to unbound start which is at the beginning of the data chain ([0,0,0,0])
w_eq_filtered_zeros.append(min(w_eq_filtered_zeros)) # serves to add the lowest weight possible to unbound start which is at the end of the data chain ([0,0,0,0])
w_eq_filtered_zeros[-2]=(min(w_eq_filtered_zeros)) # since we are transitioning into it, need to make the one before it also be the lowest

print(len(w_eq_filtered_zeros))
print(len(data))

for i in range(len(data) - 1):
    current_value = tuple(sorted(data[i]))
    next_value = tuple(sorted(data[i + 1]))
    weight = w_eq_filtered_zeros[i]  # Weight associated with this transition
    transition_matrix[value_to_index[current_value], value_to_index[next_value]] += weight*100000 # <-- this is just for abeta, used for visualizing the graphs

#x_normed = normalize(transition_matrix, axis=1, norm='l1')
x_normed = transition_matrix / transition_matrix.sum(axis=1, keepdims=True)

### Check the nature of your system: 
# you cant have sum of any row be 0 or else it means you go into a state but never come out of it, or vice versa, which isnt possible

# this is row with 1 at the diagonal position and 0 everywhere else
absorbing_states = np.where(np.all(x_normed == np.eye(x_normed.shape[0]), axis=1))[0]

if len(absorbing_states) > 0:
    print(f"Found {len(absorbing_states)} absorbing states at indices: {absorbing_states}")
else:
    print("No absorbing states found.")


For 5, there are 7426 unique pairs


7426 7426
7427 7427
7427
160295
160295


No absorbing states found.


In [ ]:
%store unique_uplets_pre_process_0

## Solving for states at equilibrium

In [8]:

print("calcaulting eigenvalues")
eigenvalues, eigenvectors = eig(x_normed.T)
equilibrium_eigenvector_index=[i for i, e in enumerate(eigenvalues) if np.isclose(e, 1.00)]
equilibrium_eigenvector=(eigenvectors[:,equilibrium_eigenvector_index])
P_eq = np.abs(np.real(equilibrium_eigenvector / np.sum(equilibrium_eigenvector)))

# for x, y in zip(P_eq,frequency):
#     print(x,y)

p_eq_keys={filtered_keys[i]: P_eq[i] for i in range(len(filtered_keys))}


calcaulting eigenvalues


### Calucalting the Kd

In [9]:

kd=list(map(lambda x: (P_eq.sum()-x)/x, P_eq))
kd_keys = {filtered_keys[i]: kd[i] for i in range(len(filtered_keys))}

# populating the dictionary
matrix=transition_matrix
#matrix=x_normed
# n = matrix.shape[0] # this REMOVES THE SELF LOOPS *** VISUALIZATION PURPOSE
# matrix[range(n), range(n)] = 0

Returning the kd values to each set of quintuplets

In [10]:

ls_letters=filtered_keys

coord_dict = {}
rows = ls_letters
cols = ls_letters

dictionary_transitions={}
for idx_i,i in enumerate(rows):
    for idx_j,j in enumerate(cols):
        dictionary_transitions[i,j]=matrix[idx_i][idx_j]

dictionary_transitions_sorted={k: v for k, v in sorted(dictionary_transitions.items(), key=lambda item: item[1],reverse=True)}
# --> run until here for the hopping v gliding mechanism
merged_data = defaultdict(int)

for (key, value) in dictionary_transitions_sorted.items():
    key_tuple = tuple(key)
    merged_data[key_tuple] += value
merged_output = [(list(key), value) for key, value in merged_data.items()]

filtered_merged_output=(list(filter(lambda x: x[1] != 0, merged_output)))

filehandler_p_eq_keys = open(f"/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/{protein_name}/p_eq_keys_weighted_magnified.pckl","wb")
pickle.dump(p_eq_keys,filehandler_p_eq_keys)
filehandler_p_eq_keys.close()

filehandler_filtered_merged_output = open(f"/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/{protein_name}/filtered_merged_output_weighted_magnified.pckl","wb")
pickle.dump(filtered_merged_output,filehandler_filtered_merged_output)
filehandler_filtered_merged_output.close()

filehandler_dict = open(f"/Users/adelielouet/Documents/science/dd_proj/msm_full_model_paper/pickled_files/{protein_name}/dictionary_transitions_sorted.pckl","wb")
pickle.dump(dictionary_transitions_sorted,filehandler_dict)
filehandler_dict.close()